<a href="https://colab.research.google.com/github/AmrBr/Reverse-Dictionary/blob/main/Reverse_Dictionary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **INSTALLS**

In [ ]:
!pip install camel-tools

### **CONSTANTS**

In [ ]:
SEED = 42
SPLITS = ["train", "validation", "test"]
COLUMNS_TO_REMOVE = ['id', 'pos', 'electra', 'bertseg', 'bertmsa']
WORD = 'word'
GLOSS = 'gloss'
DS1_PATH = 'Abreekaa/Arabic-Reverse-Dictionary'
DS2_PATH = 'riotu-lab/arabic_reverse_dictionary'
CLEAN_DATA_HF_PATH = 'Abreekaa/arabic-reverse-dictionary-merged'

## **Data Preprocessing**



In [ ]:
from datasets import load_dataset

def load_data():
  dataset_1 = load_dataset(DS1_PATH)
  dataset_bulk = load_dataset(DS2_PATH)

  dataset_bulk = dataset_bulk['train']
  return dataset_1, dataset_bulk

In [ ]:
def remove_dublicates(ds1, ds2):
  seen = set()

  for split in SPLITS:
      for ex in ds1[split]:
          seen.add((ex[WORD], ex[GLOSS]))

  ds_bulk_clean = ds2.filter(lambda ex: not_duplicate(ex, seen))
  return ds_bulk_clean

def not_duplicate(ex, seen):
    return (ex[WORD], ex[GLOSS]) not in seen

def deduplicate(ds):
    seen = set()

    def is_unique(ex):
        key = (ex[WORD], ex[GLOSS])
        if key in seen:
            return False
        seen.add(key)
        return True

    return ds.filter(is_unique)

In [ ]:
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.normalize import normalize_alef_maksura_ar, normalize_alef_ar, normalize_teh_marbuta_ar

def dediacritize(ex):
  return dediac_ar(ex)

def remove_tatweel(ex):
  return ex.replace("ـ", "")

def normalize_text(ex):
  ex = normalize_alef_maksura_ar(ex)
  ex = normalize_alef_ar(ex)
  ex = normalize_teh_marbuta_ar(ex)
  return ex

In [ ]:
def clean_text(text):
    if text is None:
        return ""

    text = text.strip()
    if not text:
        return ""

    text = remove_tatweel(text)
    text = normalize_text(text)
    text = dediacritize(text)
    return text

In [ ]:
def preprocess_text(ex):
    return {
        WORD: clean_text(ex[WORD]),
        GLOSS: clean_text(ex[GLOSS]),
    }

In [ ]:
def preprocess_dataset(ds):
    ds = ds.map(preprocess_text)
    empty_count = sum(1 for ex in ds if (ex[WORD] == "" and ex[GLOSS] == ""))
    print("Empty examples: {}".format(empty_count))
    ds = ds.filter(
        lambda ex: ex[WORD] != "" and ex[GLOSS] != ""
    )
    return ds

In [ ]:
from datasets import concatenate_datasets

def merge_datasets(ds):
  return concatenate_datasets(ds)

In [ ]:
def analyze_dataset_integrity(ds, WORD_KEY='word', GLOSS_KEY='gloss'):
    splits = ['train', 'validation', 'test']

    print(f"{'Metric':<25} | {'Train':<10} | {'Val':<10} | {'Test':<10}")
    print("-" * 65)

    words = {s: set(ds[s][WORD_KEY]) for s in splits}
    pairs = {s: set(zip(ds[s][WORD_KEY], ds[s][GLOSS_KEY])) for s in splits}

    print(f"{'Total Samples':<25} | {len(ds['train']):<10} | {len(ds['validation']):<10} | {len(ds['test']):<10}")
    print(f"{'Unique Words':<25} | {len(words['train']):<10} | {len(words['validation']):<10} | {len(words['test']):<10}")
    print(f"{'Unique Pairs (W+G)':<25} | {len(pairs['train']):<10} | {len(pairs['validation']):<10} | {len(pairs['test']):<10}")

    print("\n--- Overlap Analysis (Leakage) ---")

    def print_overlap(label, set_dict):
        tr_val = len(set_dict['train'] & set_dict['validation'])
        tr_test = len(set_dict['train'] & set_dict['test'])
        print(f"{label:<25} | Train ∩ Val: {tr_val:<5} | Train ∩ Test: {tr_test}")

    print_overlap("Word Overlap", words)
    print_overlap("Full Pair Overlap", pairs)

    print("\n--- Data Integrity Insights ---")

    for s in splits:
        dupes = len(ds[s]) - len(pairs[s])
        if dupes > 0:
            print(f"{s.capitalize()}: Found {dupes} exact duplicate (Word+Gloss) rows.")

        if len(pairs[s]) > len(words[s]):
            ambiguity_count = len(pairs[s]) - len(words[s])
            print(f"{s.capitalize()}: Has {ambiguity_count} cases where one word has multiple glosses.")

    if len(pairs['test']) > 0:
        leak_pct = (len(pairs['train'] & pairs['test']) / len(pairs['test'])) * 100
        leak_pct2 = (len(pairs['validation'] & pairs['test']) / len(pairs['test'])) * 100
        print(f"\nCritical Leakage: {leak_pct:.2f}% of Test pairs are present in Training.")

In [ ]:
def build_reverse_dictionary_dataset():
  ds1, ds2 = load_data()

  # Normalize ds1
  for split in SPLITS:
      ds1[split] = ds1[split].remove_columns(COLUMNS_TO_REMOVE)
      ds1[split] = preprocess_dataset(ds1[split])
      ds1[split] = deduplicate(ds1[split])

  # Normalize ds2
  ds2 = ds2.rename_column("definition", "gloss")
  ds2 = preprocess_dataset(ds2)
  ds2 = deduplicate(ds2)

  # Remove ds2 entries already in ds1
  ds2 = remove_dublicates(ds1, ds2)

  # Merge
  dataset_full = merge_datasets([ds1["train"], ds1["test"], ds1["validation"], ds2])
  dataset_full = deduplicate(dataset_full)

  dataset_full = dataset_full.shuffle(SEED)

  train_test = dataset_full.train_test_split(test_size=0.2, seed=42)
  test_val = train_test["test"].train_test_split(test_size=0.5, seed=42)

  validation_ds = ds1["validation"]
  test_ds = ds1["test"]

  return {
      "train": train_test["train"],
      "validation": test_val["train"],
      "test": test_val["test"],
    }


In [ ]:
from datasets import DatasetDict
def save_dataset(dataset, path):
  final_ds = DatasetDict(dataset)

  final_ds.push_to_hub(path, private=True)

In [ ]:
dataset = build_reverse_dictionary_dataset()

In [ ]:
analyze_dataset_integrity(dataset)

In [ ]:
dataset

In [ ]:
save_dataset(dataset, CLEAN_DATA_HF_PATH)

## **Model Experiments**

In [ ]:
from datasets import load_dataset, concatenate_datasets
import gc

dataset = load_dataset(CLEAN_DATA_HF_PATH)
train_ds = dataset['train']
val_ds = dataset['validation']
test_ds = dataset['test']

del dataset
gc.collect()

print(f"Train examples: {len(train_ds)}, Validation: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
from collections import defaultdict

words_length = defaultdict(int)
gloss_length = defaultdict(int)
for ex in train_ds:
  words_length[len(ex[WORD].split())] += 1
  gloss_length[len(ex[GLOSS].split())] += 1
print(words_length)
print(gloss_length)

### **TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True,
    dtype=np.float32
)

X_train = vectorizer.fit_transform(train_ds["gloss"])
train_words = list(train_ds["word"])

X_test = vectorizer.transform(test_ds["gloss"])
test_words = list(test_ds["word"])

In [ ]:
def retrieve_top_k(test_vectors, train_vectors, train_words, k=5):
    similarities = cosine_similarity(test_vectors, train_vectors)
    top_k_indices = np.argsort(-similarities, axis=1)[:, :k]
    return [[train_words[i] for i in idxs] for idxs in top_k_indices]

def evaluate(predictions, gold_words):
    top1 = sum(gold_words[i] == preds[0] for i, preds in enumerate(predictions)) / len(gold_words)
    top5 = sum(gold_words[i] in preds[:5] for i, preds in enumerate(predictions)) / len(gold_words)

    mrr = 0
    for i, preds in enumerate(predictions):
        if gold_words[i] in preds:
            mrr += 1 / (preds.index(gold_words[i]) + 1)
    mrr /= len(gold_words)

    return {"Top-1": top1, "Top-5": top5, "MRR": mrr}

In [ ]:
preds = retrieve_top_k(X_test, X_train, train_words, k=5)
metrics = evaluate(preds, test_words)
print(metrics)

### **FAISS**

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

def build_faiss_index(vectors, save_path=None):
    """
    Creates a FAISS index for Cosine Similarity.
    """
    dims = vectors.shape[1]

    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(dims)

    index.add(vectors)

    if save_path:
        faiss.write_index(index, save_path)

    return index

### **Static Embeddings (FastText + FAISS)**

In [ ]:
!pip install fasttext

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
!gunzip cc.ar.300.bin.gz

In [ ]:
def get_top_k_matches(query_text, model, index, k=5):

    query_vec = model.get_sentence_vector(query_text).reshape(1, -1).astype('float32')
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    return scores[0], indices[0]

In [ ]:
def vectorize_text_list(text_list, model):

    vectors = [model.get_sentence_vector(str(text)) for text in text_list]
    return np.array(vectors).astype('float32')

In [ ]:
import fasttext
import numpy as np

model = fasttext.load_model('cc.ar.300.bin')

train_vectors = vectorize_text_list(train_ds["gloss"], model)
test_vectors = vectorize_text_list(test_ds["gloss"], model)

index = build_faiss_index(train_vectors)

print(f"Starting evaluation on {len(test_ds['word'])} samples...")

top1_count = 0
top5_count = 0
mrr_sum = 0

for i, query in enumerate(test_ds["gloss"],):
    scores, indices = get_top_k_matches(query, model, index, k=5)

    predicted_words = [train_ds[int(idx)]['word'] for idx in indices]
    correct_word = test_ds['word'][i]

    if correct_word == predicted_words[0]:
        top1_count += 1

    if correct_word in predicted_words:
        top5_count += 1
        rank = predicted_words.index(correct_word) + 1
        mrr_sum += (1.0 / rank)

print(f"Top-1 Accuracy: {top1_count / len(test_ds["gloss"],):.4f}")
print(f"Top-5 Accuracy: {top5_count / len(test_ds["gloss"],):.4f}")
print(f"MRR: {mrr_sum / len(test_ds["gloss"],):.4f}")

### **Dynamic Embeddings**

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm import tqdm
import json
from collections import defaultdict

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

MODELS = {
    'Arabic-BERT': 'asafaya/bert-base-arabic',
    'AraElectra': 'aubmindlab/araelectra-base-discriminator',
    'AraBERT': 'aubmindlab/bert-base-arabertv2',
    'CamelBERT': 'CAMeL-Lab/bert-base-arabic-camelbert-msa',
    'MARBERT': 'UBC-NLP/MARBERT',
    'MARBERTv2': 'UBC-NLP/MARBERTv2',
}

FINE_TUNE_CONFIG = {
    'batch_size': 128,
    'learning_rate': 2e-5,
    'num_epochs': 3,
    'warmup_steps': 500,
    'max_length': 128,
    'temperature': 0.07,
    'negative_sample_size': 5,
}

In [ ]:
train_ds.set_format(type="pandas")
val_ds.set_format(type="pandas")
test_ds.set_format(type="pandas")

train_ds = train_ds[:]
val_ds = val_ds[:]
test_ds = test_ds[:]

In [ ]:
test_in_vocab = test_ds[test_ds['word'].isin(train_ds['word'])].reset_index(drop=True)

**Zero Shot**

In [ ]:
class ZeroShotEvaluator:
    """Evaluate pre-trained models without fine-tuning"""

    def __init__(self, model_name, model_id):
        self.model_name = model_name
        self.model_id = model_id
        self.device = DEVICE
        self.results = {}

        print(f"\nLoading {model_name}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_id)
            self.model = AutoModel.from_pretrained(model_id).to(self.device)
            self.model.eval()
            print(f"Loaded {model_name}")
        except Exception as e:
            print(f"Failed to load {model_name}: {e}")
            self.tokenizer = None
            self.model = None

    def get_embeddings(self, texts, batch_size=32, pooling='mean'):
        """Get embeddings for a list of texts"""
        embeddings = []

        if self.model is None:
            return None

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size),
                         desc=f"Embedding {self.model_name}"):
                batch_texts = texts[i:i+batch_size]

                inputs = self.tokenizer(
                    batch_texts,
                    return_tensors='pt',
                    max_length=FINE_TUNE_CONFIG['max_length'],
                    truncation=True,
                    padding=True
                ).to(self.device)

                outputs = self.model(**inputs)

                # Different pooling strategies
                if pooling == 'mean':
                    # Mean pooling across all tokens
                    mask = inputs['attention_mask'].unsqueeze(-1).expand(
                        outputs.last_hidden_state.size()
                    ).float()
                    sum_embeddings = torch.sum(
                        outputs.last_hidden_state * mask, 1
                    )
                    sum_mask = torch.clamp(mask.sum(1), min=1e-9)
                    batch_embeddings = sum_embeddings / sum_mask
                elif pooling == 'cls':
                    # [CLS] token only
                    batch_embeddings = outputs.last_hidden_state[:, 0, :]
                elif pooling == 'max':
                    # Max pooling
                    batch_embeddings = torch.max(
                        outputs.last_hidden_state, 1
                    )[0]
                else:
                    batch_embeddings = outputs.last_hidden_state[:, 0, :]

                embeddings.append(batch_embeddings.cpu().numpy())

        return np.vstack(embeddings)

    def evaluate(self, train_data, test_data, test_iv, train_vocab, pooling='mean'):
      if self.model is None:
          return None

      # 1. Embeddings
      train_gloss_embs = self.get_embeddings(train_data['gloss'].tolist(), pooling=pooling)
      test_gloss_embs = self.get_embeddings(test_data['gloss'].tolist(), pooling=pooling)

      # 2. Build a mapping: word -> list of row indices in train_gloss_embs
      word_to_train_indices = defaultdict(list)
      for i, row in train_data.iterrows():
          word_to_train_indices[row['word']].append(i)

      # 3. Normalize embeddings for cosine similarity
      test_norm = test_gloss_embs / np.linalg.norm(test_gloss_embs, axis=1, keepdims=True)
      train_norm = train_gloss_embs / np.linalg.norm(train_gloss_embs, axis=1, keepdims=True)

      # Full similarity matrix: (n_test, n_train)
      similarities = test_norm @ train_norm.T  # (n_test, n_train_glosses)

      # 4. Collapse per-word: for each test sample, compute one score per unique train word
      #    by taking the MAX similarity across all glosses of that word
      unique_train_words = list(word_to_train_indices.keys())
      n_test = len(test_data)
      n_words = len(unique_train_words)

      # Shape: (n_test, n_unique_train_words)
      word_sim_matrix = np.zeros((n_test, n_words), dtype=np.float32)

      for j, word in enumerate(unique_train_words):
          indices = word_to_train_indices[word]
          # Max pooling across all glosses of this word
          word_sim_matrix[:, j] = similarities[:, indices].max(axis=1)

      word_to_col = {word: j for j, word in enumerate(unique_train_words)}

      # 5. Build IV set
      iv_words = set(test_iv['word'].tolist())

      # 6. Calculate ranks and metrics
      metrics = {
          'top_1': 0, 'top_5': 0, 'mrr_sum': 0.0,
          'iv_top_1': 0, 'iv_top_5': 0, 'iv_mrr_sum': 0.0
      }

      oov_count = 0
      iv_count = 0

      for i in range(n_test):
          true_word = test_data.iloc[i]['word']

          if true_word not in word_to_col:
              oov_count += 1
              continue

          target_col = word_to_col[true_word]
          target_score = word_sim_matrix[i, target_col]

          rank = int(np.sum(word_sim_matrix[i] >= target_score))
          rr = 1.0 / rank

          is_iv = test_data.iloc[i]['word'] in iv_words

          metrics['top_1']    += int(rank <= 1)
          metrics['top_5']    += int(rank <= 5)
          metrics['mrr_sum']  += rr

          if is_iv:
              iv_count += 1
              metrics['iv_top_1']   += int(rank <= 1)
              metrics['iv_top_5']   += int(rank <= 5)
              metrics['iv_mrr_sum'] += rr

      # 7. Final results
      if oov_count > 0:
          print(f"[Warning] Skipped {oov_count} OOV test words "
                f"({oov_count / n_test * 100:.1f}% of test set).")

      results = {
          'Overall': {
              'Top-1': (metrics['top_1']   / len(test_data['word'])) * 100,
              'Top-5': (metrics['top_5']   / len(test_data['word'])) * 100,
              'MRR':    metrics['mrr_sum'] / len(test_data['word']),
          },
          'In-Vocab': {
              'Top-1': (metrics['iv_top_1']   / iv_count) * 100 if iv_count > 0 else 0.0,
              'Top-5': (metrics['iv_top_5']   / iv_count) * 100 if iv_count > 0 else 0.0,
              'MRR':    metrics['iv_mrr_sum'] / iv_count         if iv_count > 0 else 0.0,
          },
          'OOV': {
              'count': oov_count,
              'rate':  (oov_count / n_test) * 100,
          }
      }

      self.results[pooling] = results
      return results

In [ ]:
def print_results(model_name, results, stage='Zero-Shot'):
    """Pretty-print results for a model"""

    print(f"\n{model_name} ({stage}):")
    print(f"  Top-1:  In-Vocab={results['In-Vocab']['Top-1']:6.2f}%  "
          f"Overall={results['Overall']['Top-1']:6.2f}%")
    print(f"  Top-5:  In-Vocab={results['In-Vocab']['Top-5']:6.2f}%  "
          f"Overall={results['Overall']['Top-5']:6.2f}%")
    print(f"  MRR: In-Vocab={results['In-Vocab']['MRR']:6.2f}  "
          f"Overall={results['Overall']['MRR']:6.2f}")

def generate_comparison_table(all_results, stage='Zero-Shot'):
    """Generate comprehensive comparison table"""
    metrics = ['Top-1', 'Top-5', 'MRR']

    print(f"\n{stage} COMPARISON:")
    print("-" * 100)

    for metric in metrics:
      print(f"\n{metric} Results:")
      print("-" * 100)
      print(f"{'Model':<20} | {'In-Vocab':<12} | {'Overall':<12}")
      print("-" * 100)

      for model_name in MODELS.keys():
          if model_name in all_results:
              results = all_results[model_name]
              print(f"{model_name:<20} | "
                    f"{results['In-Vocab'][metric]:>10.2f}% | "
                    f"{results['Overall'][metric]:>10.2f}%")

In [ ]:
all_results = defaultdict()
for model_name, model_id in MODELS.items():
  evaluator = ZeroShotEvaluator(model_name, model_id)
  results = evaluator.evaluate(
      train_ds, test_ds, test_in_vocab, train_ds['word']
  )

  if results:
      print_results(model_name, results, stage='Zero-Shot')
      all_results[model_name] = results

In [ ]:
generate_comparison_table(all_results)

**Fine Tuning**

In [ ]:
class GlossWordDataset(Dataset):
    """Dataset for fine-tuning: (gloss, true_word, negative_words)"""

    def __init__(self, train_data, tokenizer, train_vocab,
                 neg_sample_size=5, max_length=128):
        self.glosses = train_data['gloss'].tolist()
        self.words = train_data['word'].tolist()
        self.tokenizer = tokenizer
        self.word_list = list(train_vocab)
        self.neg_size = neg_sample_size
        self.max_length = max_length

    def __len__(self):
        return len(self.glosses)

    def __getitem__(self, idx):
        gloss = self.glosses[idx]
        true_word = self.words[idx]

        # Sample negative words
        neg_words = np.random.choice(
            [w for w in self.word_list if w != true_word],
            size=min(self.neg_size, len(self.word_list) - 1),
            replace=False
        )

        return {
            'gloss': gloss,
            'true_word': true_word,
            'neg_words': neg_words.tolist()
        }

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm
import numpy as np
import math
import random

class FineTuner:
    """Fine-tune models with NT-Xent contrastive loss"""

    def __init__(self, model_name, model_id, train_data, train_vocab, device=None):
        self.model_name = model_name
        self.model_id = model_id
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.train_data = train_data
        self.train_vocab = train_vocab

        # Initialize to None to prevent AttributeError on load failure
        self.tokenizer = None
        self.model = None
        self.projection = None
        self.is_loaded = False

        print(f"\nLoading {model_name} for fine-tuning...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_id)
            self.model = AutoModel.from_pretrained(model_id).to(self.device)
            hidden_size = self.model.config.hidden_size
            self.projection = nn.Linear(hidden_size, 256).to(self.device)
            self.is_loaded = True
            print(f"Loaded {model_name} on {self.device}")
        except Exception as e:
            print(f"Failed to load {model_name}: {e}")
            self.is_loaded = False

    def _pool_embeddings(self, outputs, attention_mask, pooling='mean'):
        """Centralized pooling logic to avoid code duplication."""
        if pooling == 'mean':
            mask = attention_mask.float().unsqueeze(-1)
            sum_emb = torch.sum(outputs.last_hidden_state * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            return sum_emb / sum_mask
        elif pooling == 'cls':
            return outputs.last_hidden_state[:, 0, :]
        elif pooling == 'max':
            return torch.max(outputs.last_hidden_state, dim=1)[0]
        return outputs.last_hidden_state[:, 0, :]

    def get_embeddings(self, texts, batch_size=32, pooling='mean', apply_projection=False):
        """Generate embeddings for inference/evaluation."""
        if not self.is_loaded:
            raise RuntimeError("Model failed to load. Cannot generate embeddings.")

        embeddings = []
        self.model.eval()
        if apply_projection:
            self.projection.eval()

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding {self.model_name}"):
                batch_texts = texts[i:i+batch_size]
                inputs = self.tokenizer(
                    batch_texts, return_tensors='pt', max_length=FINE_TUNE_CONFIG['max_length'],
                    truncation=True, padding=True
                ).to(self.device)

                outputs = self.model(**inputs)
                batch_embs = self._pool_embeddings(outputs, inputs['attention_mask'], pooling)

                if apply_projection:
                    batch_embs = F.normalize(self.projection(batch_embs), p=2, dim=1)

                embeddings.append(batch_embs.cpu().numpy())
        return np.vstack(embeddings)

    def contrastive_loss(self, gloss_embs, true_word_embs, neg_word_embs):
        """NT-Xent contrastive loss with correct tensor shaping."""
        pos_sim = F.cosine_similarity(gloss_embs, true_word_embs, dim=1)

        neg_sims = F.cosine_similarity(gloss_embs.unsqueeze(1), neg_word_embs, dim=2).squeeze(1)

        all_sims = torch.cat([pos_sim.unsqueeze(1), neg_sims], dim=1)
        targets = torch.zeros(all_sims.shape[0], dtype=torch.long, device=all_sims.device)

        return F.cross_entropy(all_sims / FINE_TUNE_CONFIG['temperature'], targets)

    def train_epoch(self, train_loader, optimizer, scheduler=None, scaler=None):
        """Train one epoch with mixed precision & gradient clipping."""
        self.model.train()
        self.projection.train()
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Training {self.model_name}"):
            glosses = batch['gloss']
            true_words = batch['true_word']
            neg_words_list = batch['neg_words']

            # Mixed precision context
            with torch.autocast(device_type=self.device.type, enabled=scaler is not None):
                # 1. Encode glosses
                gloss_inputs = self.tokenizer(glosses, return_tensors='pt', max_length=FINE_TUNE_CONFIG['max_length'], truncation=True, padding=True).to(self.device)

                gloss_outputs = self.model(**gloss_inputs)
                gloss_embs = self._pool_embeddings(gloss_outputs, gloss_inputs['attention_mask'])

                # 2. Encode true words
                true_word_inputs = self.tokenizer(true_words, return_tensors='pt', max_length=FINE_TUNE_CONFIG['max_length'],
                                                  truncation=True, padding=True).to(self.device)
                true_word_outputs = self.model(**true_word_inputs)
                true_word_embs = self._pool_embeddings(true_word_outputs, true_word_inputs['attention_mask'])

                # 3. Batch encode ALL negatives at once
                flat_negs = [n for sublist in neg_words_list for n in sublist]
                neg_inputs = self.tokenizer(flat_negs, return_tensors='pt', max_length=FINE_TUNE_CONFIG['max_length'],
                                            truncation=True, padding=True).to(self.device)
                neg_outputs = self.model(**neg_inputs)
                neg_embs = self._pool_embeddings(neg_outputs, neg_inputs['attention_mask'])

                # 4. Project & L2 Normalize
                gloss_proj = F.normalize(self.projection(gloss_embs), p=2, dim=1)
                true_word_proj = F.normalize(self.projection(true_word_embs), p=2, dim=1)
                neg_proj = F.normalize(self.projection(neg_embs), p=2, dim=1)

                # 5. Reshape negatives back to (batch_size, num_negs, dim)
                batch_size = len(glosses)
                num_negs = neg_proj.shape[0] // batch_size
                neg_word_embs = neg_proj.view(batch_size, num_negs, -1)

                loss = self.contrastive_loss(gloss_proj, true_word_proj, neg_word_embs)

            # Backward pass & optimization
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
            else:
                loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(self.projection.parameters(), max_norm=1.0)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad()
            if scheduler:
                scheduler.step()

            total_loss += loss.item()

        return total_loss / len(train_loader)

    def fine_tune(self, num_epochs=3, batch_size=32, use_amp=True):
        """Full fine-tuning loop with LR scheduling & optional AMP."""
        if not self.is_loaded:
            raise RuntimeError("Model failed to load. Cannot start fine-tuning.")

        # Set seeds for reproducibility
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(SEED)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

        dataset = GlossWordDataset(
            self.train_data, self.tokenizer, self.train_vocab,
            neg_sample_size=FINE_TUNE_CONFIG['negative_sample_size']
        )
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

        # Parameter groups for proper weight decay handling
        optimizer = AdamW([
            {'params': self.model.parameters(), 'lr': FINE_TUNE_CONFIG['learning_rate'], 'weight_decay': 0.01},
            {'params': self.projection.parameters(), 'lr': FINE_TUNE_CONFIG['learning_rate'], 'weight_decay': 0.0}
        ])

        total_steps = len(loader) * num_epochs
        warmup_steps = int(0.1 * total_steps)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

        # Mixed precision scaler
        scaler = torch.amp.GradScaler('cuda') if (use_amp and torch.cuda.is_available()) else None

        print(f"\nFine-tuning {self.model_name} ({num_epochs} epochs, BS={batch_size})...")
        for epoch in range(num_epochs):
            loss = self.train_epoch(loader, optimizer, scheduler, scaler)
            print(f"  Epoch {epoch+1}/{num_epochs}: Loss = {loss:.4f}")

        print(f"✓ Fine-tuning complete")
        return self.model, self.projection

    def evaluate_retrieval(self, train_data, test_data, pooling='mean'):
        """
        Evaluates gloss-to-word retrieval performance.
        Returns Top-1, Top-5, MRR, and OOV statistics.
        """
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Cannot evaluate.")

        self.model.eval()
        self.projection.eval()

        # 1. Generate embeddings
        train_gloss_embs = self.get_embeddings(
            train_data['gloss'].tolist(), pooling=pooling, apply_projection=True
        )
        test_gloss_embs = self.get_embeddings(
            test_data['gloss'].tolist(), pooling=pooling, apply_projection=True
        )

        # 2. Map each unique train word to its row indices
        word_to_indices = defaultdict(list)
        for i, word in enumerate(train_data['word']):
            word_to_indices[word].append(i)
        unique_words = list(word_to_indices.keys())
        word_to_col = {w: j for j, w in enumerate(unique_words)}

        # 3. L2 Normalize & compute similarity matrix
        train_norm = train_gloss_embs / np.linalg.norm(train_gloss_embs, axis=1, keepdims=True)
        test_norm = test_gloss_embs / np.linalg.norm(test_gloss_embs, axis=1, keepdims=True)
        sim_matrix = test_norm @ train_norm.T  # (n_test, n_train)

        # 4. Collapse to per-word similarity (max-pool across all glosses of a word)
        n_test = len(test_data)
        word_sim_matrix = np.zeros((n_test, len(unique_words)), dtype=np.float32)
        for j, word in enumerate(unique_words):
            word_sim_matrix[:, j] = sim_matrix[:, word_to_indices[word]].max(axis=1)

        # 5. Compute ranking metrics
        metrics = {'top1': 0, 'top5': 0, 'mrr': 0.0, 'iv_top1': 0, 'iv_top5': 0, 'iv_mrr': 0.0}
        iv_count = 0
        oov_count = 0
        train_word_set = set(train_data['word'].tolist())

        for i in range(n_test):
            true_word = test_data.iloc[i]['word']
            is_iv = true_word in train_word_set

            if true_word not in word_to_col:
                oov_count += 1
                continue

            target_col = word_to_col[true_word]
            target_score = word_sim_matrix[i, target_col]

            rank = int(np.sum(word_sim_matrix[i] >= target_score))
            rr = 1.0 / rank if rank > 0 else 0.0

            metrics['top1'] += int(rank == 1)
            metrics['top5'] += int(rank <= 5)
            metrics['mrr'] += rr

            if is_iv:
                iv_count += 1
                metrics['iv_top1'] += int(rank == 1)
                metrics['iv_top5'] += int(rank <= 5)
                metrics['iv_mrr'] += rr

        n_eval = len(test_data) - oov_count
        if n_eval == 0:
            print("[Warning] No in-vocabulary test samples to evaluate.")
            return {'Overall': {'Top-1': 0, 'Top-5': 0, 'MRR': 0}, 'In-Vocab': {}}

        return {
            'Overall': {
                'Top-1': (metrics['top1'] / len(test_data)) * 100,
                'Top-5': (metrics['top5'] / len(test_data)) * 100,
                'MRR': metrics['mrr'] / len(test_data),
            },
            'In-Vocab': {
                'Top-1': (metrics['iv_top1'] / iv_count) * 100 if iv_count > 0 else 0.0,
                'Top-5': (metrics['iv_top5'] / iv_count) * 100 if iv_count > 0 else 0.0,
                'MRR': metrics['iv_mrr'] / iv_count if iv_count > 0 else 0.0,
                'Count': iv_count
            },
            'OOV': {'Count': oov_count, 'Rate': (oov_count / len(test_data)) * 100}
        }

In [ ]:
import os
all_results = {}

for model_name, model_id in MODELS.items():
    print(f"\nStarting pipeline for: {model_name}")

    # 1. Initialize & Fine-tune
    tuner = FineTuner(
        model_name=model_name,
        model_id=model_id,
        train_data=train_ds,
        train_vocab=set(train_ds['word'].tolist())
    )

    if not tuner.is_loaded:
        print(f"Skipping {model_name} due to load failure.")
        continue

    model, projection = tuner.fine_tune(num_epochs=3, batch_size=FINE_TUNE_CONFIG['batch_size'])

    # 2. Evaluate
    eval_results = tuner.evaluate_retrieval(train_ds, test_ds, pooling='mean')
    print_results(model_name, eval_results, stage="Fine-Tuned")
    all_results[model_name] = eval_results

In [ ]:
generate_comparison_table(all_results)

### **LLMs**